In [1]:
import numpy as np
import pandas as pd

from mord import LogisticAT
from scipy.optimize import minimize

In [2]:
# Define MSE function
def mse(x, model, features, P_true):
    theta = x[:5]
    coeff = x[5:]
    model.theta_ = theta
    model.coef_ = coeff
    P_est = model.predict_proba(features)
    return np.sum((P_true - P_est) ** 2)

In [3]:
# Load training data
df_train = pd.read_csv("data/data_train.zip")
embedding_matrix_train = np.array([eval(embedding) for embedding in df_train["embeddings_512"]])

In [4]:
# Load constrained score distributions (one per training sample)
P_fair = np.load("data/P_constrained_js.npz")["arr_0"]

In [5]:
# Load base model trained without fairness constraints
model_unconstrained = np.load("data/model_unconstrained.npz")

In [6]:
# Initialize constrained model
model_constrained = LogisticAT(alpha=5e-3)
model_constrained.theta_ = model_unconstrained["theta"]
model_constrained.coef_ = model_unconstrained["coef"]

In [7]:
# Fit constrained model to score distributions (this can take time)
res = minimize(
    mse, 
    np.concat((model_constrained.theta_, model_constrained.coef_)), 
    args=(model_constrained, embedding_matrix_train, P_fair), 
    method="L-BFGS-B", options={"maxfun": 100000000}
)

In [8]:
# Inspect outcome
res

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 286.66442001326425
        x: [-1.658e+02  1.018e+02 ...  2.625e+01 -5.576e+00]
      nit: 3976
      jac: [-2.728e-04  6.594e-04 ...  5.684e-06 -9.095e-05]
     nfev: 2309244
     njev: 4458
 hess_inv: <517x517 LbfgsInvHessProduct with dtype=float64>

In [9]:
# Save model
# np.savez("data/model_constrained_js.npz", theta=model_constrained.theta_, coef=model_constrained.coef_)